# 基于 Vision Transformer 的 CIFAR-10 图像分类实验

这个 notebook 版本复用 `exp4` 目录下的模型、数据、训练、评估和绘图模块，完整流程包括：加载 CIFAR-10、训练 CNN baseline、训练 ViT、测试集评估、保存图表和汇总指标。

如果服务器网络无法下载 CIFAR-10，可以手动下载 `cifar-10-python.tar.gz`，上传到仓库根目录的 `data/` 下，然后运行 notebook 里的“CIFAR-10 数据准备”单元格完成 MD5 校验和解压。

## 1. 环境与导入

In [ ]:
from __future__ import annotations

import csv
import sys
from datetime import datetime
from pathlib import Path

import torch
from torch import nn
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "exp4":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from exp4.evaluate import evaluate_model
from exp4.models import SimpleCNN, build_vit_model
from exp4.train import resolve_device, save_json, seed_everything, train_model
from exp4.utils.data import build_cifar10_dataloaders
from exp4.utils.cifar10_download import (
    CIFAR10_ARCHIVE,
    CIFAR10_ARCHIVE_MD5,
    CIFAR10_URL,
    ensure_cifar10_data,
    file_md5,
    has_cifar10_files,
)
from exp4.utils.metrics import count_parameters
from exp4.utils.plot import plot_accuracy_curve, plot_loss_curve, plot_model_comparison

print(f"Project root: {PROJECT_ROOT}")

## 2. 实验配置

默认使用完整 CIFAR-10。调试时可以把 `train_subset`、`val_subset`、`test_subset` 改成小整数，例如 `256`、`64`、`64`。

In [ ]:
RUN_CONFIG = {
    "seed": 42,
    "device": "auto",
    "data_root": PROJECT_ROOT / "data",
    "output_root": PROJECT_ROOT / "exp4" / "outputs",
    "image_size": 224,
    "val_fraction": 0.1,
    "num_workers": 2,
    "download": True,
    "download_timeout_seconds": 60,
    "use_tensorboard": True,
    "train_subset": None,
    "val_subset": None,
    "test_subset": None,
}

MODEL_CONFIGS = {
    "cnn": {
        "epochs": 5,
        "batch_size": 16,
        "lr": 1e-3,
        "weight_decay": 1e-4,
    },
    "vit": {
        "epochs": 3,
        "batch_size": 8,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "pretrained": True,
        "train_mode": "head_only",  # "head_only" or "full"
    },
}

RUN_CONFIG, MODEL_CONFIGS

## 3. CIFAR-10 数据准备（手动下载入口）

服务器网络不好时，先在网络好的机器下载 `cifar-10-python.tar.gz`，上传到 `data/` 目录，然后运行下面单元格。它会校验 MD5 并解压为 `data/cifar-10-batches-py/`。

In [ ]:
PREPARE_CIFAR10_NOW = True
DOWNLOAD_FROM_NOTEBOOK_IF_MISSING = False  # 服务器网络差时保持 False，手动上传归档更稳

data_root = Path(RUN_CONFIG["data_root"])
archive_path = data_root / CIFAR10_ARCHIVE

print(f"CIFAR-10 URL: {CIFAR10_URL}")
print(f"Expected MD5: {CIFAR10_ARCHIVE_MD5}")
print(f"Data root: {data_root}")
print(f"Archive path: {archive_path}")

if PREPARE_CIFAR10_NOW:
    if has_cifar10_files(data_root):
        print(f"CIFAR-10 is already extracted under: {data_root / 'cifar-10-batches-py'}")
    elif archive_path.exists():
        actual_md5 = file_md5(archive_path)
        print(f"Found archive: {archive_path}")
        print(f"Actual MD5: {actual_md5}")
        ensure_cifar10_data(
            data_root=data_root,
            download=False,
            timeout_seconds=RUN_CONFIG["download_timeout_seconds"],
            archive_path=archive_path,
        )
    elif DOWNLOAD_FROM_NOTEBOOK_IF_MISSING:
        ensure_cifar10_data(
            data_root=data_root,
            download=True,
            timeout_seconds=RUN_CONFIG["download_timeout_seconds"],
        )
    else:
        display(Markdown(
            "未发现 CIFAR-10 数据或归档。请先下载并上传：\n\n"
            f"- 下载地址：`{CIFAR10_URL}`\n"
            f"- 上传位置：`{archive_path}`\n"
            f"- 期望 MD5：`{CIFAR10_ARCHIVE_MD5}`\n\n"
            "上传后重新运行本单元格即可自动校验和解压。"
        ))

## 4. 工具函数

In [ ]:
def create_experiment_dirs(output_root: Path) -> dict[str, Path]:
    experiment_name = datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_dir = output_root / experiment_name
    paths = {
        "experiment": experiment_dir,
        "figures": experiment_dir / "figures",
        "checkpoints": experiment_dir / "checkpoints",
        "logs": experiment_dir / "logs",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    (output_root / "latest_experiment.txt").write_text(str(experiment_dir.resolve()), encoding="utf-8")
    return paths


def build_model(model_name: str) -> nn.Module:
    if model_name == "cnn":
        return SimpleCNN(num_classes=10)
    if model_name == "vit":
        config = MODEL_CONFIGS["vit"]
        return build_vit_model(
            num_classes=10,
            pretrained=config["pretrained"],
            train_mode=config["train_mode"],
        )
    raise ValueError(f"Unsupported model: {model_name}")


def write_comparison_csv(results: dict[str, dict], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        "model",
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "parameters",
        "trainable_parameters",
        "best_val_accuracy",
    ]
    with output_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for model_name, result in results.items():
            writer.writerow({"model": model_name, **result})

## 5. 初始化实验目录

In [ ]:
seed_everything(RUN_CONFIG["seed"])
device = resolve_device(RUN_CONFIG["device"])
paths = create_experiment_dirs(Path(RUN_CONFIG["output_root"]))

save_json(
    {
        "run_config": RUN_CONFIG,
        "model_configs": MODEL_CONFIGS,
        "device": str(device),
        "experiment_dir": str(paths["experiment"]),
    },
    paths["logs"] / "notebook_run_config.json",
)

print(f"Device: {device}")
print(f"Experiment directory: {paths['experiment']}")

## 6. 单模型训练与评估函数

In [ ]:
def run_one_model(model_name: str) -> dict:
    model_config = MODEL_CONFIGS[model_name]
    print(
        f"Loading CIFAR-10 for {model_name} "
        f"from {RUN_CONFIG['data_root']} (download={RUN_CONFIG['download']})",
        flush=True,
    )
    loaders = build_cifar10_dataloaders(
        data_root=RUN_CONFIG["data_root"],
        batch_size=model_config["batch_size"],
        num_workers=RUN_CONFIG["num_workers"],
        image_size=RUN_CONFIG["image_size"],
        val_fraction=RUN_CONFIG["val_fraction"],
        seed=RUN_CONFIG["seed"],
        download=RUN_CONFIG["download"],
        pin_memory=device.type == "cuda",
        train_subset=RUN_CONFIG["train_subset"],
        val_subset=RUN_CONFIG["val_subset"],
        test_subset=RUN_CONFIG["test_subset"],
        download_timeout=RUN_CONFIG["download_timeout_seconds"],
    )
    print(f"{model_name} data | train={loaders.train_size} val={loaders.val_size} test={loaders.test_size}")

    model = build_model(model_name)
    total_parameters = count_parameters(model)
    trainable_parameters = count_parameters(model, trainable_only=True)
    print(f"{model_name} parameters: total={total_parameters:,}, trainable={trainable_parameters:,}")

    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=model_config["lr"],
        weight_decay=model_config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(1, model_config["epochs"]),
    )

    training = train_model(
        model=model,
        train_loader=loaders.train_loader,
        val_loader=loaders.val_loader,
        device=device,
        model_name=model_name,
        epochs=model_config["epochs"],
        optimizer=optimizer,
        scheduler=scheduler,
        checkpoint_dir=paths["checkpoints"],
        log_dir=paths["logs"],
        criterion=nn.CrossEntropyLoss(),
        use_tensorboard=RUN_CONFIG["use_tensorboard"],
    )
    plot_loss_curve(
        training.history,
        paths["figures"] / f"{model_name}_loss_curve.png",
        title=f"{model_name.upper()} loss curve",
    )
    plot_accuracy_curve(
        training.history,
        paths["figures"] / f"{model_name}_acc_curve.png",
        title=f"{model_name.upper()} accuracy curve",
    )

    evaluation = evaluate_model(
        model=model,
        data_loader=loaders.test_loader,
        device=device,
        class_names=loaders.class_names,
        model_name=model_name,
        checkpoint_path=training.best_checkpoint,
        figures_dir=paths["figures"],
        log_dir=paths["logs"],
        criterion=nn.CrossEntropyLoss(),
        seed=RUN_CONFIG["seed"],
    )

    return {
        "accuracy": evaluation["accuracy"],
        "macro_precision": evaluation["macro_precision"],
        "macro_recall": evaluation["macro_recall"],
        "macro_f1": evaluation["macro_f1"],
        "parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "best_val_accuracy": training.best_val_accuracy,
    }

## 7. 训练 CNN Baseline

In [ ]:
results = {}
results["cnn"] = run_one_model("cnn")
results["cnn"]

## 8. 训练 Vision Transformer

In [ ]:
results["vit"] = run_one_model("vit")
results["vit"]

## 9. 保存对比结果

In [ ]:
save_json(results, paths["logs"] / "notebook_comparison_summary.json")
write_comparison_csv(results, paths["logs"] / "notebook_comparison_summary.csv")
plot_model_comparison(results, paths["figures"] / "notebook_model_comparison.png")

display(Markdown("### CNN 与 ViT 对比"))
for model_name, result in results.items():
    print(
        f"{model_name.upper()} | acc={result['accuracy']:.4f} "
        f"macro_f1={result['macro_f1']:.4f} params={result['parameters']:,}"
    )
print(f"Outputs saved to: {paths['experiment']}")

## 10. 展示结果图

In [ ]:
figure_names = [
    "cnn_loss_curve.png",
    "cnn_acc_curve.png",
    "cnn_confusion_matrix.png",
    "cnn_prediction_samples.png",
    "vit_loss_curve.png",
    "vit_acc_curve.png",
    "vit_confusion_matrix.png",
    "vit_prediction_samples.png",
    "notebook_model_comparison.png",
]

for figure_name in figure_names:
    figure_path = paths["figures"] / figure_name
    if figure_path.exists():
        display(Markdown(f"### {figure_name}"))
        display(Image(filename=str(figure_path)))